# Weekly S2S anomaly skill (weeks 1-8)

## Purpose

Subseasonal skill of the five initialization configurations following the ESP-Lab S2S protocol (`ESP-Lab/jupyter/s2s_skill`, `esp_lab/diagnostics/s2s_core.py`): weekly lead blocks, anomalies from separate model and observed climatologies, ACC with significance, and the skill difference between initialization methods.

## Inputs

As `01_compute_and_plot_lead_skill.ipynb`: daily member fields of every configuration and start date, ERA5/IMERG/NOAA OLR references and their climatology over `CLIM_YEARS`, and the monthly output of the free-running E3SMv3 AMIP runs `AMIP_CASES` (`AMIP_ROOT/<case>/post/{atm,lnd}/180x360_aave/`) for the model climatology.

## Method

`util/fcst_init_skill.py`:

1. Weeks 1-8 = lead days 1-7, ..., 50-56 (7-day means; ensemble mean for the model).
2. Anomalies on the verification days (`util/fcst_init_data.py`). Observed: reference minus the reference climatology of `CLIM_YEARS`; for the 00Z pressure-level fields the calendar-day climatology of the 00Z states (`SMOOTH_DAYS` running mean), for the daily-mean fields the monthly climatology interpolated linearly between mid-month values (ERA5 monthly product corrected per calendar month to the sampling of the 6-hourly archive, 2011-2012; IMERG and NOAA OLR monthly means of their daily series). Forecast (`BIAS = "amip"`): ensemble mean minus the monthly climatology of the AMIP runs over the same years, interpolated the same way. With 4 starts per season the hindcast's own lead-dependent climatology is unavailable; the AMIP climatology removes the model's mean bias without using observations of the verified period, and the initialization drift stays in the forecast error. For the 00Z fields the AMIP monthly mean covers all hours (no sub-daily T/U/Z output), a fixed pattern in the forecast anomaly only (about 0.5 K for T850, 0.5 m s-1 for U850 globally). `BIAS = "loo"` (mean ensemble-mean error over the other starts) is kept for comparison: starts 5 days apart verify on overlapping days, so it leaks their observed anomalies into the forecast anomaly.
3. Pattern ACC (centred, cos-lat weighted) per region, start and week, averaged over starts; MSSS against the climatology forecast.
4. Gridpoint ACC over the start dates of the available seasons pooled (8 samples with both seasons: one-sided t-test critical value 0.62 at 95%; 4 samples with one: 0.90; stippled), and its difference from `BASELINE` over the seasons both configurations have.

Caveat: ERA5 is the verifying analysis and the source of the HYB_BASE/DA_LND atmospheric initial state, which favours them in week 1.

## Outputs

`data/hind_lead_eval/s2s_weekly/`: `clim_<SEASON>_<VAR>.nc` (reference), `clim_model_<SEASON>_<VAR>.nc` (AMIP), `anom_<CONFIG>_<SEASON>_<VAR>.nc` (weekly anomalies), `s2s_weekly_scores.csv` (pattern ACC, MSSS, RMSE per week); figures in `figure/hind_lead_eval/s2s_weekly/`: weekly regional scores, gridpoint ACC maps (2 x 4 weeks) per configuration, ΔACC maps vs `BASELINE`, and the pattern-ACC scorecard.

## 1. Libraries

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

## 2. User setup

Paths and run options. Edit this cell for normal changes; it holds plain values only, so `papermill -p` overrides work.

In [2]:
INPUT_DATA_ROOT = Path("/global/cfs/cdirs/m4849/zhan391/eam_hindcast")
WORK_DIR = Path("/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/data")  # reference-data archive
LAND_SUPPORT_DIR = OBSERVATION_ROOT / "lnd_sea_mask"           # land fraction, ELM layer thicknesses
WORKFLOW_NAME = "hind_lead_eval"

# Initialization configurations (configs/fcst_init_experiment_config.py) and seasons (4 starts each)
CONFIGS = ("CLIM_BASE", "HYB_BASE", "DA_ATM", "DA_LND", "DA_COUPLED")
SEASONS = ("DJF", "JJA")
MAX_LEAD = 56          # lead days (8 weeks)
MAX_MEMBERS = None     # e.g. 4 for a quick test
REQUIRE_ALL_MEMBERS = True  # skip a (configuration, season) until every member's stream is regenerated

VARIABLES = ("Z500", "T850", "U850", "U200", "TMQ", "PSL", "TREFHT", "PRECT", "FLUT")
REGIONS = ("GLOBAL", "LAND", "OCEAN", "NH_POLAR", "NINO3.4", "MAR_CONT", "CONUS")
CLIM_YEARS = (2001, 2014)  # common to ERA5, IMERG (from 2001) and the AMIP monthly output (FLUT to 2014)
SMOOTH_DAYS = 31           # running mean of the daily reference climatology (00Z fields)
BIAS = "amip"              # "amip" (forecast anomaly from the AMIP climatology), "none" or "loo" (comparison only)
AMIP_ROOT = OBSERVATION_ROOT / "E3SMv3"                 # free-running E3SMv3 DECK AMIP runs
AMIP_CASES = ("v3.LR.amip_0101", "v3.LR.amip_0151")
BASELINE = "HYB_BASE"      # ΔACC and scorecard reference
MAP_VARIABLES = ("TREFHT", "PRECT", "Z500")
PLOT_REGIONS = ("GLOBAL", "LAND")

FORCE_COMPUTE = False
SHOW = True
FIGURE_DPI = 200

## 3. Paths and modules

In [3]:
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.fcst_init_experiment_config import MEMBERS, check_configs, init_dates, readiness
from configs.output_paths import workflow_output_dirs

check_configs(CONFIGS)
data_dir, figure_dir = workflow_output_dirs(WORKFLOW_NAME, "s2s_weekly", output_root=DIAGNOSTIC_OUTPUT_ROOT)
ROOTS = dict(data_root=INPUT_DATA_ROOT, land_support_dir=LAND_SUPPORT_DIR)
print(f"[DATA] {data_dir}\n[FIGURE] {figure_dir}")


def members_of(config):
    return MEMBERS[config] if MAX_MEMBERS is None else MEMBERS[config][:MAX_MEMBERS]


def save(fig, name):
    path = figure_dir / name
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"[FIGURE] {path}")
    if not SHOW:
        plt.close(fig)

[DATA] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly
[FIGURE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/figure/hind_lead_eval/s2s_weekly


## 4. Read input

In [4]:
status = readiness(CONFIGS, SEASONS, streams=(("atm", "daily"), ("atm", "6hourly")), data_root=INPUT_DATA_ROOT)
summary = status.groupby(["config", "season"]).agg(ready=("ready", "sum"), members=("members", "sum"))
display(summary.unstack("season"))
RUNS = [(c, s) for (c, s), row in summary.iterrows() if row.ready >= (row.members if REQUIRE_ALL_MEMBERS else 1)]
print(f"[RUNS] {len(RUNS)} of {len(summary)} configuration-seasons ready: {RUNS}")

ready      members     
season       DJF  JJA     DJF  JJA
config                            
CLIM_BASE    160  160     160  160
DA_ATM       160   56     160  160
DA_COUPLED   160   56     160  160
DA_LND       160   56     160  160
HYB_BASE     160  160     160  160

[RUNS] 7 of 10 configuration-seasons ready: [('CLIM_BASE', 'DJF'), ('CLIM_BASE', 'JJA'), ('DA_ATM', 'DJF'), ('DA_COUPLED', 'DJF'), ('DA_LND', 'DJF'), ('HYB_BASE', 'DJF'), ('HYB_BASE', 'JJA')]


## 5. Derive and save

Reference climatology per season and variable (read once, saved), then the weekly anomalies of each configuration. Saved files are reused unless `FORCE_COMPUTE`.

In [ ]:
from util.fcst_init_data import model_climatology, open_campaign, open_reference, reference_climatology
from util.fcst_init_skill import weekly_anomalies

years = list(range(CLIM_YEARS[0], CLIM_YEARS[1] + 1))
anom_path = lambda c, s, v: data_dir / f"anom_{c}_{s}_{v}.nc"


def cached_climatology(path, build):
    """Saved climatology of the current ``CLIM_YEARS``, else built and saved."""
    if path.exists() and not FORCE_COMPUTE:
        clim = xr.load_dataarray(path)
        if clim.attrs.get("climatology_years") == f"{years[0]}-{years[-1]}":
            return clim
    clim = build()
    clim.to_netcdf(path)
    print(f"[WRITE] {path}  ({clim.attrs.get('climatology_source', '')})")
    return clim


def anom_current(path):
    """An anomaly file made with the current ``BIAS`` and ``CLIM_YEARS``."""
    if not path.exists():
        return False
    with xr.open_dataset(path) as ds:
        return ds.attrs.get("bias") == BIAS and ds.attrs.get("clim_years") == str(CLIM_YEARS)


for season in SEASONS:
    inits = init_dates([season])[season]
    for var in VARIABLES:
        todo = [c for c in CONFIGS if (c, season) in RUNS and (FORCE_COMPUTE or not anom_current(anom_path(c, season, var)))]
        if not todo:
            continue
        try:
            ref = open_reference(var, inits, max_lead=MAX_LEAD, observation_root=OBSERVATION_ROOT).load()
            clim = cached_climatology(data_dir / f"clim_{season}_{var}.nc", lambda: reference_climatology(
                var, inits, years, max_lead=MAX_LEAD, smooth_days=SMOOTH_DAYS, observation_root=OBSERVATION_ROOT))
            mclim = cached_climatology(data_dir / f"clim_model_{season}_{var}.nc", lambda: model_climatology(
                var, inits, years, max_lead=MAX_LEAD, amip_root=AMIP_ROOT, cases=AMIP_CASES,
                land_support_dir=LAND_SUPPORT_DIR)) if BIAS == "amip" else None
        except (FileNotFoundError, KeyError) as err:
            print(f"[SKIP] {var}: {err}")
            continue
        for c in todo:
            model = open_campaign(c, season, var, members=members_of(c), max_lead=MAX_LEAD, **ROOTS).load()
            fa, oa = weekly_anomalies(model, ref, clim, bias=BIAS, model_climatology=mclim)
            xr.Dataset({"model": fa, "reference": oa}).assign_attrs(
                bias=BIAS, clim_years=str(CLIM_YEARS), reference_climatology=clim.attrs.get("climatology_source", ""),
                model_climatology="" if mclim is None else mclim.attrs.get("climatology_source", "")).to_netcdf(anom_path(c, season, var))
            print(f"[WRITE] {anom_path(c, season, var)}")
            del model
anom = {(c, s, v): xr.load_dataset(anom_path(c, s, v)) for c in CONFIGS for s in SEASONS for v in VARIABLES
        if anom_path(c, s, v).exists()}
print(f"[LOAD] {len(anom)} anomaly files")

[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/clim_DJF_Z500.nc  (ERA5 daily, 31-day running mean)
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/clim_model_DJF_Z500.nc  (E3SMv3 AMIP monthly (v3.LR.amip_0101, v3.LR.amip_0151))
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_CLIM_BASE_DJF_Z500.nc
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_HYB_BASE_DJF_Z500.nc
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_DA_ATM_DJF_Z500.nc
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_DA_LND_DJF_Z500.nc
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_DA_COUPLED_DJF_Z500.nc
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3

/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis/diagnostic/util/fcst_init_data.py:248: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 124. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(files, combine="by_coords", chunks={"time": 124}, data_vars="minimal",
/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis/diagnostic/util/fcst_init_data.py:248: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 124. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(files, combine="by_coords", chunks={"time": 124}, data_vars="minimal",
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'TMQ' has multiple fill values {np.float32(-32767.0), np.float32(-9999.0)} defined, decoding all va

[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/clim_DJF_TMQ.nc  (ERA5 monthly + 6-hourly sampling offset (2011+2012))
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/clim_model_DJF_TMQ.nc  (E3SMv3 AMIP monthly (v3.LR.amip_0101, v3.LR.amip_0151))
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_CLIM_BASE_DJF_TMQ.nc
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_HYB_BASE_DJF_TMQ.nc
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_DA_ATM_DJF_TMQ.nc
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_DA_LND_DJF_TMQ.nc
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_DA_COUPLED_DJF_TMQ.nc


/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'PSL' has multiple fill values {np.float32(-32767.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/clim_DJF_PSL.nc  (ERA5 monthly + 6-hourly sampling offset (2011+2012))
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/clim_model_DJF_PSL.nc  (E3SMv3 AMIP monthly (v3.LR.amip_0101, v3.LR.amip_0151))
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_CLIM_BASE_DJF_PSL.nc
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_HYB_BASE_DJF_PSL.nc
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_DA_ATM_DJF_PSL.nc
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_DA_LND_DJF_PSL.nc
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_DA_COUPLED_DJF_PSL.nc


/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/xarray/conventions.py:205: SerializationWarning: variable 'TREFHT' has multiple fill values {np.float32(-32767.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/clim_DJF_TREFHT.nc  (ERA5 monthly + 6-hourly sampling offset (2011+2012))
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/clim_model_DJF_TREFHT.nc  (E3SMv3 AMIP monthly (v3.LR.amip_0101, v3.LR.amip_0151))
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_CLIM_BASE_DJF_TREFHT.nc
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/s2s_weekly/anom_HYB_BASE_DJF_TREFHT.nc


## 6. Scores

Regional pattern ACC (mean over starts) and MSSS per week; gridpoint ACC over the starts of the available seasons and its difference from `BASELINE` (on the seasons common to both).

In [ ]:
from configs.s2s_config import TCC_REGIONS
from util.fcst_init_data import region_masks
from util.fcst_init_skill import critical_acc, gridpoint_acc, msss, pattern_acc, pool_seasons

regions = [r for r in TCC_REGIONS if r["name"] in REGIONS]
masks = None
rows = []
for (c, s, v), ds in anom.items():
    if masks is None:
        masks = region_masks(regions, ds.lat.values, ds.lon.values, land_support_dir=LAND_SUPPORT_DIR)
    pa = pattern_acc(ds.model, ds.reference, masks).mean("init")
    sk = msss(ds.model, ds.reference, masks)
    for r in masks.region.values:
        for w in ds.week.values:
            rows.append(dict(config=c, season=s, variable=v, region=r, week=int(w),
                             pattern_acc=float(pa.sel(region=r, week=w)), msss=float(sk.msss.sel(region=r, week=w)),
                             rmse_anom=float(sk.rmse_anom.sel(region=r, week=w)),
                             rmse_clim=float(sk.rmse_clim.sel(region=r, week=w))))
table = pd.DataFrame(rows)
table.to_csv(data_dir / "s2s_weekly_scores.csv", index=False)
print(f"[WRITE] {data_dir / 's2s_weekly_scores.csv'}")
if len(table):
    display(table[table.region == "GLOBAL"].pivot_table(index=["variable", "config"], columns=["season", "week"],
                                                        values="pattern_acc", sort=False).round(2))

def seasons_of(c, v):
    return [s for s in SEASONS if (c, s, v) in anom]


def pooled_acc(c, v, seasons):
    fa = pool_seasons({s: anom[c, s, v].model for s in seasons})
    oa = pool_seasons({s: anom[c, s, v].reference for s in seasons})
    return gridpoint_acc(fa, oa).assign_attrs(seasons="+".join(seasons))


gp_acc = {(c, v): pooled_acc(c, v, seasons_of(c, v)) for c in CONFIGS for v in VARIABLES if seasons_of(c, v)}
for c in CONFIGS:
    n = sorted({a.attrs["n_samples"] for (cc, _), a in gp_acc.items() if cc == c})
    if n:
        print(f"[ACC] {c}: gridpoint ACC over {n} starts; 95% one-sided critical value "
              f"{', '.join(f'{critical_acc(k):.2f}' if k > 2 else 'n/a' for k in n)}")

## 7. Plot and show

In [ ]:
from util.fcst_init_plotting import plot_config_week_maps, plot_scorecard, plot_week_maps, plot_weekly_regional
from util.fcst_init_skill import relative_change

for s in SEASONS:
    for r in PLOT_REGIONS:
        if len(table[(table.season == s) & (table.region == r)]):
            save(plot_weekly_regional(table, VARIABLES, region=r, season=s), f"s2s_weekly_{s}_{r}.png")

for (c, v), acc in gp_acc.items():
    if v in MAP_VARIABLES:
        n = acc.attrs["n_samples"]
        r_crit = critical_acc(n) if n > 2 else np.nan
        save(plot_week_maps(acc, title=f"{c} {v} weekly ACC ({n} starts, {acc.attrs['seasons']}; stippled > {r_crit:.2f})",
                            stipple=acc > r_crit, label="ACC"), f"s2s_acc_map_{c}_{v}.png")
for v in MAP_VARIABLES:
    base_seasons = seasons_of(BASELINE, v)
    diff, pooled = {}, set()
    for c in CONFIGS:
        common = [s for s in seasons_of(c, v) if s in base_seasons]
        if c == BASELINE or not common:
            continue
        # same starts for both, so a configuration missing a season is not compared with a larger sample
        diff[c] = pooled_acc(c, v, common) - pooled_acc(BASELINE, v, common)
        pooled.add("+".join(common))
    if not diff:
        continue
    for w in (2, 4):
        save(plot_config_week_maps(diff, week=w, title=f"{v} ΔACC vs {BASELINE}, week {w} (common starts: {', '.join(sorted(pooled))})",
                                   levels=np.linspace(-0.6, 0.6, 13), label="ΔACC"), f"s2s_dacc_{v}_week{w}.png")

for s in SEASONS:
    t = table[(table.season == s) & (table.region == "GLOBAL")]
    if BASELINE not in set(t.config):
        continue
    wide = t.pivot_table(index="config", columns=["variable", "week"], values="pattern_acc", sort=False)
    wide = wide.reindex(index=[c for c in CONFIGS if c in wide.index])
    save(plot_scorecard(relative_change(wide, BASELINE, "pattern_acc"), higher_is_better=True,
                        title=f"{s} global pattern ACC minus {BASELINE}", units="ΔACC"), f"s2s_scorecard_{s}.png")